# **1. Instalacción y preparación de Detecctron2**

In [ ]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.2/274.2 kB 14.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Cloning into 'detectron2'...
remote: Enumerating objects: 15900, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 15900 (delta 55), reused 21 (delta 21), pack-reused 15802 (from 4)
Receiving objects: 100% (15900/15900), 6.45 MiB | 5.17 MiB/s, done.
Resolving deltas: 100% (11564/11564), done.
Ignoring dataclasses: markers 'python_version < "3.7"' don't match your env

In [ ]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
torch:  2.6 ; cuda:  cu124
detectron2: 0.6


In [ ]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

# **2. Preparar datasets**
## **2.1. Cargar Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install geopandas rasterio shapely pyproj fiona matplotlib

## **2.2. Crear máscaras binarias para modelos de segmentación**

In [ ]:
import os
import rasterio
from rasterio import features
import fiona
import numpy as np

def crear_mascara_semantica(img_path, shp_path, salida_path):
    # Leer la imagen base para obtener metadatos
    with rasterio.open(img_path) as src:
        meta = src.meta.copy()
        shape = src.shape  # (altura, ancho)
        transform = src.transform
        crs = src.crs

    # Leer los polígonos del shapefile
    with fiona.open(shp_path, "r") as shapefile:
        geometries = [feature["geometry"] for feature in shapefile]

    # Crear máscara binaria: 1 si pertenece a un arbusto, 0 si es fondo
    mask = features.rasterize(
        ((geom, 1) for geom in geometries),  # Asignar valor 1 a todos los arbustos
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype=rasterio.uint8
    )

    # Guardar la máscara como imagen GeoTIFF
    meta.update({
        "count": 1,
        "dtype": "uint8"
    })

    with rasterio.open(salida_path, "w", **meta) as dst:
        dst.write(mask, 1)

def procesar_directorio(img_dir, shp_dir, salida_dir):
    os.makedirs(salida_dir, exist_ok=True)
    for nombre in os.listdir(img_dir):
        if nombre.endswith(".tif") and nombre.startswith("Img_"):
            numero = nombre.replace("Img_", "").replace(".tif", "")
            img_path = os.path.join(img_dir, f"Img_{numero}.tif")
            shp_path = os.path.join(shp_dir, f"Pol_{numero}.shp")
            salida_path = os.path.join(salida_dir, f"Mask_{numero}.tif")
            if os.path.exists(shp_path):
                print(f"Generando máscara semántica para Img_{numero}...")
                crear_mascara_semantica(img_path, shp_path, salida_path)
            else:
                print(f"No se encontró el shapefile para Img_{numero}")
